# RAG – Retrieval Augmented Generation

## Solution
### CIML Summer Institute
### UC San Diego



## Setup: LangChain

In this RAG tutorial, we'll be working with [LangChain](https://python.langchain.com/v0.2/docs/introduction/), which is a powerful framework for building applications with language models. LangChain provides utilities for working with various language model providers, integrating embeddings, and creating chains for more complex applications. Below are the necessary imports for this notebook:

In [1]:
import os
import random
import glob
import numpy as np
import warnings

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_classic.memory import ConversationSummaryMemory
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Ollama
from langchain_community.document_loaders import TextLoader
from chromadb.utils import embedding_functions
from langchain_community.embeddings import HuggingFaceEmbeddings

warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/scratch/mhnguyen/job_51495100/ipykernel_1305820/2345354522.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


# Part 1: Retrieval

- In this section, we'll focus on the retrieval aspect of RAG. We'll start by understanding vectorization, followed by storing and retrieving vectors efficiently.

## Vectorizing

- **Vectorization** is the process of converting text into vectors in an embedding space. These vectors capture the semantic meaning of the text, enabling us to perform various operations like similarity calculations. We'll use HuggingFaceEmbeddings for this task. You can see the documentation for this langchain object [here](https://api.python.langchain.com/en/latest/embeddings/langchain_community.embeddings.huggingface.HuggingFaceEmbeddings.html).


In [2]:
vectorizer = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

This vectorizer converts text into vectors in embedding space. Lets try seeing how we can use this.

In [3]:
vectorizer.embed_query("dog")[0:10]

[-0.053146980702877045,
 0.014194361865520477,
 0.007145709823817015,
 0.06860866397619247,
 -0.0784803256392479,
 0.010167473927140236,
 0.10228310525417328,
 -0.012064892798662186,
 0.09521345049142838,
 -0.030350156128406525]

- As you can see from above, this converts text into a series of numbers. 

### Task 1

**Your job is to write a function that takes in two strings, vectorize them, and return their cosine similarity.** Implement the following function.

#### `similarity_two_queries`

In [4]:
# SOLUTION
def similarity_two_queries(word1, word2):
    word1_vec = vectorizer.embed_query(word1)
    word2_vec = vectorizer.embed_query(word2)
    return np.dot(word1_vec,word2_vec)

- Observe the similarity scores of both **'cat'** and **'dog'** to the word **'kitten'**

In [5]:
print("Similarity of 'kitten' and 'cat': ",similarity_two_queries("kitten","cat"))
print("Similarity of 'kitten' and 'dog': ",similarity_two_queries("kitten","dog"))

Similarity of 'kitten' and 'cat':  0.7882107784673581
Similarity of 'kitten' and 'dog':  0.5205050705569765


- By using the previously defined function,  we can take pairs of texts and quantify how **similar** they are.

### Task 2

**Which of the following words in the list `words` are most related to the word 'color'?** The function `similarity_list` takes a list of words, and outputs the word and similarity score from highest to lowest. 

In [6]:
def similarity_list(word,words):
    similarity_list = [(i,similarity_two_queries("color",i)) for i in words]
    sorted_similarity_list = sorted(similarity_list,key=lambda x:x[1],reverse=True)
    return sorted_similarity_list

In [7]:
words = ["rainbow","car","black","red","cat","tree"]

In [8]:
# SOLUTION
similarity_list("color",words)

[('black', np.float64(0.785572006438183)),
 ('red', np.float64(0.7491880202776512)),
 ('rainbow', np.float64(0.5601087559108393)),
 ('car', np.float64(0.404062734299458)),
 ('cat', np.float64(0.3583904765748241)),
 ('tree', np.float64(0.35735521543042864))]

### Task 3

Each query below has an appropriate text that allows you to answer the question. The function `match_queries_with_texts` matches a query with its most related text. **Come up with 3 more questions and 3 suitable answers and add them to the list below.**

In [9]:
def match_queries_with_texts(queries, texts):
    # Calculate similarities between each query and text
    similarities = np.zeros((len(queries), len(texts)))
    
    for i, query in enumerate(queries):
        for j, text in enumerate(texts):
            similarities[i, j] = similarity_two_queries(query, text)
    
    # Match each query to the text with the highest similarity
    matches = {}
    for i, query in enumerate(queries):
        best_match_idx = np.argmax(similarities[i])
        matches[query] = texts[best_match_idx]
    
    return matches

In [10]:
#SOLUTION
queries = ["What are the 7 colors of the rainbow?", 
           "What does Elsie do for work?", 
           "Which country has the largest population?",
           "What time is it?",
           "What is the largest continent?",
           "Who is the greatest Football player?"]
texts = ["China has 1.4 billion people.",
         "Elsie works the register at Arby's.", 
         "The colors of the rainbow are ROYGBIV.",
         "The time is 3:14.",
         "The largest continent is Asia.",
         "Christiano Ronaldo"]

- Now we shuffle the queries and texts. Let's see if we can match them!

In [11]:
import random
random.shuffle(queries)
random.shuffle(texts)

match_queries_with_texts(queries, texts)

{'Which country has the largest population?': 'China has 1.4 billion people.',
 'What does Elsie do for work?': "Elsie works the register at Arby's.",
 'What are the 7 colors of the rainbow?': 'The colors of the rainbow are ROYGBIV.',
 'What time is it?': 'The time is 3:14.',
 'What is the largest continent?': 'The largest continent is Asia.',
 'Who is the greatest Football player?': 'Christiano Ronaldo'}

## Vector Store

- Now lets look at how we can store these for efficient retrieval of the vectors. There are many options for storage but in this exercise, we use [ChromaDB](https://python.langchain.com/v0.1/docs/integrations/vectorstores/chroma/)
 which is an open-source vector DB.

Through langchain, we can set the database to be a langchain ***retriever*** object, which essentially allows us to perform queries similarly to what we have done before.

- **Taking the `texts` and `queries` that you defined before, we can load it into ChromaDB and similarly perform the same operations.**

In [12]:
ids = list(range(len(texts)))
random_id = random.randint(100000, 999999)
db = Chroma.from_texts(texts, vectorizer, metadatas=[{"id": id} for id in ids],collection_name=f"temp_{random_id}")
retriever = db.as_retriever(search_kwargs={"k": 1})

In [13]:
texts

['The time is 3:14.',
 'Christiano Ronaldo',
 'China has 1.4 billion people.',
 "Elsie works the register at Arby's.",
 'The colors of the rainbow are ROYGBIV.',
 'The largest continent is Asia.']

In [14]:
retriever.invoke("Which country has the largest population?")

[Document(metadata={'id': 2}, page_content='China has 1.4 billion people.')]

- `workplaces.txt` contains names and workplaces of several people. Now let’s apply the same retrieval process to a file we read in.


In [15]:
with open("workplaces.txt", 'r') as file:
    lines = file.readlines()
lines = [line.strip() for line in lines]
print(lines[0:4])

["Aaron works at McDonald's", 'Beth works at Starbucks', 'Charlie works at Walmart', 'Daisy works at Amazon']


`workplace_retriever` is a function that takes in the workplace.txt file and returns a database as retriever that you can use to find out the workplaces of people in the file. You can specify the top-k results in the argument of the function.

In [16]:
def workplace_retriever(k=3):
    with open("workplaces.txt", 'r') as file:
        lines = file.readlines()
    lines = [line.strip() for line in lines]
    
    db = Chroma.from_texts(
        lines,
        vectorizer,
        metadatas=[{"id": id} for id in range(len(lines))],
        collection_name=f"temp_{id(lines)}"
    )
    
    retriever = db.as_retriever(search_kwargs={"k": k})
    return retriever

### Task 4

Using `workplace_retriever`, **find out who works at Starbucks and McDonald's**.

In [17]:
# SOLUTION
workplace_retriever(3).invoke("Who works at starbucks")

[Document(metadata={'id': 27}, page_content='Brian works at Starbucks'),
 Document(metadata={'id': 1}, page_content='Beth works at Starbucks'),
 Document(metadata={'id': 0}, page_content="Aaron works at McDonald's")]

In [18]:
# SOLUTION
workplace_retriever(3).invoke("Who works at McDonald's")

[Document(metadata={'id': 0}, page_content="Aaron works at McDonald's"),
 Document(metadata={'id': 26}, page_content="Alice works at McDonald's"),
 Document(metadata={'id': 22}, page_content='Wendy works at Reddit')]

## Chunking

The `workplaces.txt` data we just looked at was conveniently split into lines, with each line representing a distinct and meaningful chunk of information. This straightforward structure makes it easier to process and analyze the text data.

However, it is usually not so straightforward:
- When dealing with text data, especially from large or complex documents, it's essential to handle the formatting and structure efficiently.
- If we get a not-so-simply formatted file, we can break it down into manageable chunks using LangChain's `TextLoader` and `RecursiveCharacterTextSplitter`.
- This allows us to preprocess and chunk the data effectively for further use in our RAG pipeline.

Lets take a look at some of the *Expanse* documentation [here](https://www.sdsc.edu/support/user_guides/expanse.html). We have downloaded the contents of this webpage into two text files named `expanse_doc_1.txt` and `expanse_doc_2.txt`.

In [19]:
with open("expanse_doc_1.txt", 'r') as file:
    lines = file.readlines()
lines = [line.strip() for line in lines]
print(lines[20:35])

['Job Charging', 'Compiling', 'Running Jobs', 'GPU Nodes', 'Data Movement', 'Storage', 'Composable Systems', 'Software Packages', 'Publications', 'Expanse User Guide', 'Technical Summary', '', '', 'Expanse is a dedicated Advanced Cyberinfrastructure Coordination Ecosystem: Services and Support (ACCESS) cluster designed by Dell and SDSC delivering 5.16 peak petaflops, and will offer Composable Systems and Cloud Bursting.', '']


- We see that the data and text is not split into meaningful chunks of information by default, so we need to try out best to format it in such a way it can be useful. This is why we use chunks, which capture local and neighboring texts, grouping them together.

- When using the RecursiveCharacterTextSplitter, the chunk size determines the maximum size of each text chunk. This is particularly useful when dealing with large documents that need to be split into smaller, manageable pieces for better retrieval and analysis.

In [20]:
def expanse_retriever(chunk_size):
    loader = TextLoader('expanse_doc_1.txt')
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=10, separators=[" ", ",", "\n"])
    texts = text_splitter.split_documents(documents)
    db = Chroma(embedding_function=vectorizer, collection_name=f"expanse_temp_{id(texts)}")
    db.add_documents(texts)
    retriever = db.as_retriever(search_kwargs={"k": 3})
    return retriever

### Task 5

A function that chunks `expanse_doc_1.txt` has been provided above, experiment with different chunk sizes and pick a size that captures enough information to answer the question: ***"How do you run jobs on expanse?"*** Try sizes **10, 100 and 1000** and observe what info is being given.

In [21]:
# SOLUTION
expanse_retriever(1000).invoke("How do you run jobs on expanse?")

[Document(metadata={'source': 'expanse_doc_1.txt'}, page_content='up to 30M core-hours.\nJob Scheduling Policies\nThe maximum allowable job size on Expanse is 4,096 cores – a limit that helps shorten wait times since there are fewer nodes in idle state waiting for large number of nodes to become free.\nExpanse supports long-running jobs - run times can be extended to one week. Users requests will be evaluated based on number of jobs and job size. \nExpanse supports shared-node jobs (more than one job on a single node). Many applications are serial or can only scale to a few cores. Allowing shared nodes improves job throughput, provides higher overall system utilization, and allows more users to run on Expanse.\nTechnical Details\nSystem Component\tConfiguration\nCompute Nodes\nCPU Type\tAMD EPYC 7742\nNodes\t728\nSockets\t2\nCores/socket\t64\nClock speed\t2.25 GHz\nFlop speed\t4608 GFlop/s\nMemory capacity\t\n* 256 GB DDR4 DRAM\n\nLocal Storage\t\n1TB Intel P4510 NVMe PCIe SSD\n\nMax C

## Multiple Document Chunking

When we have more than one document we want to use in our database, we can simply iteratively chunk them. Metadata for the text source is added by default, but we can add our own metadata as well in the form of IDs.


### Task 6

`expanse_all_retriever` is a function that chunks both `expanse_doc_1.txt` and `expanse_doc_2.txt` has been provided below, using a chunk size of 1000 characters, **find** which document information for ***"Compiling Codes"*** is most likely to be located. *Hint: Look at the metadata*

In [22]:
def expanse_all_retriever(chunk_size):
    random_id = random.randint(100000, 999999)  # random 6-digit ID for uniqueness

    db = Chroma(
        embedding_function=vectorizer,
        collection_name=f"expanse_all_temp_{random_id}"
    )

    pattern = 'expanse_doc_*.txt'
    file_list = glob.glob(pattern)

    for file_name in file_list:
        loader = TextLoader(file_name)
        documents = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=10,
            separators=[" ", ",", "\n"]
        )
        texts = text_splitter.split_documents(documents)

        for i, text in enumerate(texts):
            text.metadata["chunk_number"] = i

        db.add_documents(texts)

    retriever = db.as_retriever(search_kwargs={"k": 3})
    return retriever

In [23]:
chunks = expanse_all_retriever(1000).invoke("Compiling Codes")
for chunk in chunks:
    print(chunk.metadata)

# The answer is expanse_doc_2.txt

{'source': 'expanse_doc_2.txt', 'chunk_number': 0}
{'source': 'expanse_doc_2.txt', 'chunk_number': 1}
{'chunk_number': 5, 'source': 'expanse_doc_2.txt'}


## Part 2: Basic RAG


Ollama is an open-source LLM platform that provides local access to different open-weights LLMs.

In a terminal window in Jupyter Lab, do the following to start an Ollama instance:

```export OLLAMA_MODELS=/cm/shared/examples/sdsc/ciml/2026/ollama_models/```

```export OLLAMA_PORT="$(shuf -i 3000-65000 -n 1)" && echo "${OLLAMA_PORT}" > ~/.ollama_port```

```export OLLAMA_HOST="127.0.0.1:${OLLAMA_PORT}"```

```ollama serve > ollama.log 2>&1 &```

In [24]:
import os
from ollama import Client

MODEL = "gemma4:e4b"

# Read the port from the file
with open(os.path.expanduser('~/.ollama_port')) as f:
     port = f.read().strip()

# Connect to 127.0.0.1:<port>
host = f"http://127.0.0.1:{port}"

client = Client(host=host)

In [25]:
MODEL = "gemma4:e4b" 

In [26]:
llm = Ollama(
    model=MODEL,
    base_url=f"http://127.0.0.1:{port}",  # CRITICAL: Use your custom port
    temperature=0
)

In [27]:
llm.invoke("Who are you?")

'I am Gemma 4, an open weights Large Language Model developed by Google DeepMind.\n\nI am designed to process and understand human language, as well as images, and generate helpful, informative, and creative text responses.\n\nYou can ask me to:\n*   Answer questions on a wide range of topics.\n*   Summarize complex texts.\n*   Write creative content (stories, poems, scripts).\n*   Help with coding and technical explanations.\n*   Engage in general conversation.\n\nHow can I help you today?'

### Task 7

**Write a function that uses the `workplace_retriever` function to parse your question, retrieves relevant responses from `workplace_retriever`, and then sends this context to the LLM for it to answer your question in natural language.** Fill in `workplace_question` which accomplishes this task.

In [28]:
#SOLUTION
def workplace_question(question):
    retriever = workplace_retriever()
    context = retriever.invoke(question)
    llm = Ollama(model=MODEL,base_url=f"http://127.0.0.1:{port}",temperature="0.2")
    prompt = f"Based on the following context: {context}, answer the question: "
    response = llm.invoke(prompt + question)
    return response

In [29]:
print(workplace_question("Who are the people that work at Starbucks?"))

Brian and Beth work at Starbucks.


## Part 3: LangChain RAG

The above is a very simple example of a RAG. Now, using langchain, we can put everything together in a cleaner and all inclusive way in one go. Let's combine everything we've learned into the function `generate_rag`.

- The below implementation has a custom class that allows us to view what chunks are being used based on our queries.

In [30]:
def generate_rag(verbose=False, chunk_info=False):
    import glob
    random_id = random.randint(100000, 999999)
    vectorizer = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    db = Chroma(embedding_function=vectorizer, collection_name=f"expanse_all_temp_{random_id}")
    pattern = 'expanse_doc_*.txt'
    file_list = glob.glob(pattern)
    for file_name in file_list:
        loader = TextLoader(file_name)
        documents = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=10, separators=[" ", ",", "\n"])
        texts = text_splitter.split_documents(documents)
        for id,text in enumerate(texts):
            text.metadata["chunk_number"] = id
        db.add_documents(texts)
    
    template = """<s>[INST] Given the context - {context} </s>[INST] [INST] Answer the following question - {question}[/INST]"""
    pt = PromptTemplate(
                template=template, input_variables=["context", "question"]
            )
    # Let's retrieve the top 3 chunks for our results
    retriever = db.as_retriever(search_kwargs={"k": 3})
    class CustomRetrievalQA(RetrievalQA):
        def invoke(self, *args, **kwargs):
            result = super().invoke(*args, **kwargs)
            if chunk_info:
                # Print out the chunks that were retrieved
                print("Chunks being looked at:")
                chunks = retriever.invoke(*args, **kwargs)
                for chunk in chunks:
                    print(f"Source: {chunk.metadata['source']}, Chunk number: {chunk.metadata['chunk_number']}")
                    print(f"Text snippet: {chunk.page_content[:200]}...\n")  # Print the first 200 characters
            return result
    rag = CustomRetrievalQA.from_chain_type(
        llm=Ollama(model=MODEL,base_url=f"http://127.0.0.1:{port}", temperature="0"),
        retriever=retriever,
        memory=ConversationSummaryMemory(llm=Ollama(model=MODEL, base_url=f"http://127.0.0.1:{port}")),
        chain_type_kwargs={"prompt": pt, "verbose": verbose},
    )

    return rag

### Task 8
**Compare how gemma performs without context, and with context**, so without RAG and with RAG.

In [31]:
print(llm.invoke("How can a user check their resource allocations and the resources they can use on the Expanse supercomputer"))

Since I do not have real-time access to the Expanse supercomputer's specific internal management system, I cannot provide the exact command or link. However, I can provide a comprehensive guide covering the standard methods used across almost all High-Performance Computing (HPC) environments.

Checking resource allocations usually involves a combination of checking your **job queue status**, your **account limits**, and the **system's overall capacity**.

Here is a step-by-step guide on how to check your resources, starting with the easiest methods.

---

### 💻 Method 1: The Web Portal (Easiest)

Most modern supercomputers use a web-based dashboard or job scheduler interface. This is the easiest place to start.

1.  **Check the Cluster Dashboard:** Look for a dedicated web portal for Expanse (e.g., `https://expanse.university.edu/hpc/dashboard`).
2.  **Job Queue View:** Log in with your credentials. There should be a section labeled **"Job Queue," "My Jobs,"** or **"Resource Usage."** 

In [32]:
expanse_rag = generate_rag()
result = expanse_rag.invoke("How can a user check their resource allocations and the resources they can use on the Expanse supercomputer")
print(result["result"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Based on the provided context, a user can check their resource allocations and the resources they can use on Expanse using the `expanse-client` command.

Here are the specific steps and commands:

### 1. Check Project Resource Allocations (Usage)

To review the available projects and resource usage for a specific resource, use the `user` parameter and the `-r` flag:

*   **Command Syntax:** `expanse-client user -r [resource_name]`
*   **Example:** `expanse-client user -r expanse`
*   **Purpose:** This command displays a table showing the `USED` and `AVAILABLE` resources for the specified project/resource.

### 2. Check the Full List of Available Resources

To see a comprehensive list of all available resources on the system, use the `resource` command:

*   **Command:** `expanse-client resource`
*   **Purpose:** This command provides the full list of resources that can be utilized.

***

**Note:** Before running these commands, the context shows that the user should first load the nece

- We can see what is exactly being passed into the LLM highlighted in green when we set `verbose` to True.

In [33]:
expanse_rag = generate_rag(verbose=True)
result = expanse_rag.invoke("How can a user check their resource allocations and the resources they can use on the Expanse supercomputer")
print(result["result"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
<s>[INST] Given the context - Compute Units (SSCUs), comprising 728 standard nodes, 54 GPU nodes and 4 large-memory nodes. Every Expanse node has access to a 12 PB Lustre parallel file system (provided by Aeon Computing) and a 7 PB Ceph Object Store system. Expanse uses the Bright Computing HPC Cluster management system and the SLURM workload manager for job scheduling.

Expanse Portal Login

Expanse supports the ACCESS core software stack, which includes remote login, remote computation, data movement, science workflow support, and science gateway support toolkits.

Expanse is an NSF-funded system operated by the San Diego Supercomputer Center at UC San Diego, and is available through the ACCESS program.

Resource Allocation Policies
The maximum allocation for a Principle Investigator on Expanse is 15M core-hours and 75K GPU hours. Limiting the allocation size means that Expanse c

- For more concise information, the function defined allows us to see individual chunk details as well as their source.

In [34]:
expanse_rag = generate_rag(chunk_info=True)
result = expanse_rag.invoke("How can a user check their resource allocations and the resources they can use on the Expanse supercomputer")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks being looked at:
Source: expanse_doc_1.txt, Chunk number: 1
Text snippet: Compute Units (SSCUs), comprising 728 standard nodes, 54 GPU nodes and 4 large-memory nodes. Every Expanse node has access to a 12 PB Lustre parallel file system (provided by Aeon Computing) and a 7 P...

Source: expanse_doc_1.txt, Chunk number: 2
Text snippet: up to 30M core-hours.
Job Scheduling Policies
The maximum allowable job size on Expanse is 4,096 cores – a limit that helps shorten wait times since there are fewer nodes in idle state waiting for lar...

Source: expanse_doc_1.txt, Chunk number: 12
Text snippet: script provides additional details regarding project availability and usage.  The script is located at:

/cm/shared/apps/sdsc/current/bin/expanse-client

The script uses the 'sdsc' module, which is lo...



In [35]:
print(result["result"])

Based on the provided context, a user can check their resource allocations and the resources they can use on Expanse using the `expanse-client` command.

Here are the specific steps and commands:

### 1. Check Project Resource Allocations (Usage)

To review the available projects and resource usage for a specific resource, use the `user` parameter and the `-r` flag:

*   **Command Syntax:** `expanse-client user -r [resource_name]`
*   **Example:** `expanse-client user -r expanse`
*   **Purpose:** This command displays a table showing the `USED` and `AVAILABLE` resources for the specified project/resource.

### 2. Check the Full List of Available Resources

To see a comprehensive list of all available resources on the system, use the `resource` command:

*   **Command:** `expanse-client resource`
*   **Purpose:** This command provides the full list of resources that can be utilized.

***

**Note:** Before running these commands, the context shows that the user should first load the nece

Great work! We've officially made a chatbot that can help us out with all things *Expanse*, at least according to the two .txt files we have access to!